# Muon internals — 3 ablations in one notebook

Same from-scratch setup as Part 1 (HF model random-initialized from config, trained on WikiText-2),
but here we probe **how Muon works**, not whether it beats AdamW. Three cheap experiments:

1. **Newton-Schulz steps** — `ns_steps ∈ {0,1,2,3,5}`. `0` = no orthogonalization (just normalized
   momentum), which isolates whether the *orthogonalization* matters or it's "just momentum." Plus
   the wall-clock you buy back by using fewer steps.
2. **LR-robustness basin** — sweep the learning rate ~10–16× for Muon and AdamW, plot final loss vs
   LR. The claim: Muon's basin is flat; AdamW falls off a cliff.
3. **Effective rank over training** — track the *stable rank* of weight matrices. The claim:
   AdamW-trained matrices drift to low rank; Muon's orthogonalized updates keep them high-rank.

**Notes:** Gemma is gated (`huggingface-cli login`); both use `trust_remote_code`. These are short
runs — toggle `RUN` / models / steps in the config to control cost. Single seed.

In [ ]:
# --- Setup -------------------------------------------------------------------
import importlib, subprocess, sys
for pkg in ("datasets", "transformers", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", pkg], check=True)

import math, time
from contextlib import nullcontext
import torch, torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
def amp():
    return torch.autocast("cuda", dtype=torch.bfloat16) if use_bf16 else nullcontext()
print("torch", torch.__version__, "| device:", device, "| bf16:", use_bf16,
      "|", torch.cuda.get_device_name(0) if device == "cuda" else "")
torch.backends.cuda.matmul.allow_tf32 = True
# from huggingface_hub import login; login()   # Gemma is gated

In [ ]:
# --- Config ------------------------------------------------------------------
MODELS = {
    "gemma3-270m": dict(repo="google/gemma-3-270m", trust_remote_code=True),
    "lfm2.5-350m": dict(repo="LiquidAI/LFM2.5-350M", trust_remote_code=True),
}

RUN = [1, 2, 3]   # which ablations to run

CFG = dict(
    block_size = 256, batch_size = 16,
    muon_lr = 0.02, adam_lr = 3e-4, weight_decay = 0.0,   # wd=0 to isolate the optimizer (esp. for rank)
    warmup = 50, min_lr_frac = 0.1, grad_clip = 1.0,
    eval_every = 100, eval_iters = 20, seed = 1337,
    # exp 1 — NS steps
    ns_list = [0, 1, 2, 3, 5], ns_steps_count = 600,
    # exp 2 — LR basin
    muon_lr_list = [0.005, 0.01, 0.02, 0.04, 0.08],
    adam_lr_list = [1e-4, 3e-4, 1e-3, 3e-3],
    basin_steps = 400,
    # exp 3 — effective rank
    rank_steps = 1000,
)
CFG

## Reference Muon (with `ns_steps` and aspect scaling)

Same quintic Newton-Schulz as Part 1, with `ns_steps` exposed for the ablation. `ns_steps=0` skips
the iterations entirely → the update is just the normalized (momentum) gradient, no orthogonalization.
Scaling is `aspect` (`max(1, rows/cols)**0.5`) since these are full weight matrices, not LoRA factors.

In [ ]:
# --- Reference Muon ----------------------------------------------------------
def zeropower_via_newtonschulz5(G, steps=5):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    transposed = X.size(0) > X.size(1)
    if transposed:
        X = X.T
    X = X / (X.norm() + 1e-7)
    for _ in range(steps):          # steps=0 -> no orthogonalization, just normalized direction
        A = X @ X.T
        B = b * A + c * (A @ A)
        X = a * X + B @ X
    if transposed:
        X = X.T
    return X

class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95, weight_decay=0.0, ns_steps=5, scale_mode="aspect"):
        super().__init__(params, dict(lr=lr, momentum=momentum, weight_decay=weight_decay,
                                      ns_steps=ns_steps, scale_mode=scale_mode))
    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, mom = group["lr"], group["momentum"]
            wd, ns, sm = group["weight_decay"], group["ns_steps"], group["scale_mode"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                st = self.state[p]
                if "m" not in st:
                    st["m"] = torch.zeros_like(g)
                buf = st["m"]; buf.mul_(mom).add_(g)
                g = g.add(buf, alpha=mom)
                u = zeropower_via_newtonschulz5(g, ns)
                if wd:
                    p.mul_(1 - lr * wd)
                scale = (max(1.0, p.size(0) / p.size(1)) ** 0.5) if sm == "aspect" else 1.0
                p.add_(u.to(p.dtype), alpha=-lr * scale)

print("Muon ready.")

In [ ]:
# --- Data: WikiText-2 --------------------------------------------------------
from datasets import load_dataset
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

def build_data(tok):
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
    enc = lambda s: torch.tensor(tok("\n".join(ds[s]["text"]))["input_ids"], dtype=torch.long)
    tr, va = enc("train"), enc("validation")
    print(f"  tokens: train={len(tr):,}  val={len(va):,}")
    return tr, va

def get_batch(data, bs, bl):
    ix = torch.randint(len(data) - bl - 1, (bs,))
    x = torch.stack([data[i:i + bl] for i in ix])
    return x.to(device)

In [ ]:
# --- Model + optimizers + train + metrics ------------------------------------
def fresh_model(spec):
    cfg = AutoConfig.from_pretrained(spec["repo"], trust_remote_code=spec["trust_remote_code"])
    if hasattr(cfg, "use_cache"):
        cfg.use_cache = False
    torch.manual_seed(CFG["seed"])   # identical init across runs
    return AutoModelForCausalLM.from_config(
        cfg, trust_remote_code=spec["trust_remote_code"]).to(device)

def classify_params(model):
    embed = set()
    for m in model.modules():
        if isinstance(m, nn.Embedding):
            embed.add(id(m.weight))
    for n, m in model.named_modules():
        if n.endswith("lm_head") and getattr(m, "weight", None) is not None:
            embed.add(id(m.weight))
    hidden, rest = [], []
    for _, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (hidden if (p.ndim == 2 and id(p) not in embed) else rest).append(p)
    return hidden, rest

def build_optims(model, which, lr, ns_steps=5):
    hidden, rest = classify_params(model)
    if which == "adamw":
        opts = [torch.optim.AdamW(hidden + rest, lr=lr, betas=(0.9, 0.95),
                                  weight_decay=CFG["weight_decay"])]
    else:  # muon on 2D hidden matrices; AdamW (fixed lr) on embeddings/norms
        opts = [Muon(hidden, lr=lr, momentum=0.95, weight_decay=CFG["weight_decay"], ns_steps=ns_steps),
                torch.optim.AdamW(rest, lr=CFG["adam_lr"], betas=(0.9, 0.95))]
    for o in opts:
        for g in o.param_groups:
            g["base_lr"] = g["lr"]
    return opts

def lr_frac(step, total):
    if step < CFG["warmup"]:
        return step / max(1, CFG["warmup"])
    if step >= total:
        return CFG["min_lr_frac"]
    prog = (step - CFG["warmup"]) / max(1, total - CFG["warmup"])
    return CFG["min_lr_frac"] + (1 - CFG["min_lr_frac"]) * 0.5 * (1 + math.cos(math.pi * prog))

@torch.no_grad()
def estimate_loss(model, data):
    model.eval(); tot = 0.0
    for _ in range(CFG["eval_iters"]):
        x = get_batch(data, CFG["batch_size"], CFG["block_size"])
        with amp():
            tot += model(input_ids=x, labels=x).loss.item()
    model.train(); return tot / CFG["eval_iters"]

@torch.no_grad()
def stable_rank(model, k=6):
    # stable rank = ||W||_F^2 / ||W||_2^2  (in [1, true_rank]); mean over k sampled hidden matrices.
    srs = []
    for n, p in model.named_parameters():
        if p.ndim == 2 and not any(x in n.lower() for x in ("embed", "wte", "wpe", "lm_head")):
            W = p.detach().float()
            spec = torch.linalg.matrix_norm(W, ord=2)
            srs.append(((W * W).sum() / (spec * spec + 1e-12)).item())
            if len(srs) >= k:
                break
    return sum(srs) / max(len(srs), 1)

def train_run(spec, which, train_data, val_data, lr, ns_steps=5, steps=None, log_rank=False):
    steps = steps or CFG["ns_steps_count"]
    model = fresh_model(spec)
    opts = build_optims(model, which, lr, ns_steps)
    model.train()
    log, t0 = [], time.time()
    bs, bl = CFG["batch_size"], CFG["block_size"]
    for step in range(1, steps + 1):
        frac = lr_frac(step, steps)
        for o in opts:
            for g in o.param_groups:
                g["lr"] = g["base_lr"] * frac
        x = get_batch(train_data, bs, bl)
        for o in opts:
            o.zero_grad(set_to_none=True)
        with amp():
            loss = model(input_ids=x, labels=x).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        for o in opts:
            o.step()
        if step % CFG["eval_every"] == 0 or step == steps:
            rec = dict(step=step, val_loss=estimate_loss(model, val_data), sec=time.time() - t0)
            if log_rank:
                rec["stable_rank"] = stable_rank(model)
            log.append(rec)
    sps = (time.time() - t0) / steps
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return log, sps

In [ ]:
# --- Run the ablations -------------------------------------------------------
results = {}
for name, spec in MODELS.items():
    print(f"\n{'='*60}\n{name}  ({spec['repo']})\n{'='*60}")
    try:
        tok = AutoTokenizer.from_pretrained(spec["repo"], trust_remote_code=spec["trust_remote_code"])
        if tok.pad_token is None and tok.eos_token is not None:
            tok.pad_token = tok.eos_token
        tr, va = build_data(tok)
        R = {}

        if 1 in RUN:
            print(" [1] Newton-Schulz step ablation")
            R["ns"] = {}
            for ns in CFG["ns_list"]:
                log, sps = train_run(spec, "muon", tr, va, CFG["muon_lr"], ns_steps=ns,
                                     steps=CFG["ns_steps_count"])
                best = min(e["val_loss"] for e in log)
                R["ns"][ns] = dict(best=best, ms_per_step=sps * 1000)
                print(f"   ns={ns}: best val {best:.3f} (ppl {math.exp(min(best,20)):.1f})  "
                      f"{sps*1000:.0f} ms/step")

        if 2 in RUN:
            print(" [2] LR-robustness basin")
            R["basin"] = {"muon": {}, "adamw": {}}
            for lr in CFG["muon_lr_list"]:
                log, _ = train_run(spec, "muon", tr, va, lr, steps=CFG["basin_steps"])
                R["basin"]["muon"][lr] = min(e["val_loss"] for e in log)
                print(f"   muon  lr={lr:<7}: best {R['basin']['muon'][lr]:.3f}")
            for lr in CFG["adam_lr_list"]:
                log, _ = train_run(spec, "adamw", tr, va, lr, steps=CFG["basin_steps"])
                R["basin"]["adamw"][lr] = min(e["val_loss"] for e in log)
                print(f"   adamw lr={lr:<7}: best {R['basin']['adamw'][lr]:.3f}")

        if 3 in RUN:
            print(" [3] Effective (stable) rank over training")
            R["rank"] = {}
            for which, lr in [("adamw", CFG["adam_lr"]), ("muon", CFG["muon_lr"])]:
                log, _ = train_run(spec, which, tr, va, lr, steps=CFG["rank_steps"], log_rank=True)
                R["rank"][which] = [(e["step"], e["stable_rank"]) for e in log]
                print(f"   {which}: stable rank {R['rank'][which][0][1]:.1f} -> "
                      f"{R['rank'][which][-1][1]:.1f}")

        results[name] = R
        del tr, va
    except Exception as e:
        print(f"  SKIPPED {name}: {type(e).__name__}: {e}")
print("\ndone.")

In [ ]:
# --- Plots -------------------------------------------------------------------
import matplotlib.pyplot as plt
DISPLAY = {"gemma3-270m": "Gemma 3 270M", "lfm2.5-350m": "LFM2.5 350M"}
models = [m for m in results if results.get(m)]

# Exp 1: best val + ms/step vs ns_steps
if any("ns" in results[m] for m in models):
    fig, ax = plt.subplots(figsize=(7, 4.6)); ax2 = ax.twinx()
    for m in models:
        if "ns" not in results[m]:
            continue
        ns = sorted(results[m]["ns"])
        ax.plot(ns, [results[m]["ns"][n]["best"] for n in ns], "-o", label=f"{DISPLAY[m]} val")
        ax2.plot(ns, [results[m]["ns"][n]["ms_per_step"] for n in ns], "--s", alpha=0.5)
    ax.set_xlabel("Newton-Schulz steps"); ax.set_ylabel("best val loss")
    ax2.set_ylabel("ms / step (dashed)"); ax.set_title("How much orthogonalization do you need?")
    ax.grid(alpha=0.3); ax.legend(); plt.tight_layout()
    plt.savefig("ablation_ns_steps.png", dpi=150, bbox_inches="tight"); plt.show()

# Exp 2: LR basin — best val vs LR (log x), Muon vs AdamW
if any("basin" in results[m] for m in models):
    fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 4.6), squeeze=False)
    for ax, m in zip(axes[0], models):
        if "basin" not in results[m]:
            continue
        for arm, col in [("muon", "tab:blue"), ("adamw", "tab:red")]:
            d = results[m]["basin"][arm]
            xs = sorted(d)
            ax.plot(xs, [d[x] for x in xs], "-o", color=col, label=arm)
        ax.set_xscale("log"); ax.set_xlabel("learning rate"); ax.set_ylabel("best val loss")
        ax.set_title(f"LR robustness — {DISPLAY[m]}"); ax.grid(alpha=0.3, which="both"); ax.legend()
    plt.tight_layout(); plt.savefig("ablation_lr_basin.png", dpi=150, bbox_inches="tight"); plt.show()

# Exp 3: stable rank over training
if any("rank" in results[m] for m in models):
    fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 4.6), squeeze=False)
    for ax, m in zip(axes[0], models):
        if "rank" not in results[m]:
            continue
        for arm, col in [("muon", "tab:blue"), ("adamw", "tab:red")]:
            pts = results[m]["rank"][arm]
            ax.plot([s for s, _ in pts], [r for _, r in pts], "-o", color=col, label=arm)
        ax.set_xlabel("step"); ax.set_ylabel("mean stable rank")
        ax.set_title(f"Weight-matrix rank — {DISPLAY[m]}"); ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout(); plt.savefig("ablation_rank.png", dpi=150, bbox_inches="tight"); plt.show()

import json
with open("results_ablations.json", "w") as f:
    json.dump(results, f)
print("saved plots + results_ablations.json")

## What to look for

1. **NS steps:** does val loss *plateau* by 2–3 steps (i.e. you don't need 5)? And how much ms/step
   do fewer steps save? `ns=0` should be clearly worse — that's the proof the *orthogonalization*
   (not just momentum) is doing the work.
2. **LR basin:** Muon's curve should stay flat across the sweep while AdamW's shoots up at the
   wrong LRs ("falls off a cliff"). If AdamW is flat too, the claim doesn't hold at this scale.
3. **Stable rank:** if AdamW's mean stable rank drifts *down* over training while Muon's stays
   higher, that's the "orthogonal updates preserve rank" result. If they track together, report
   that honestly.

Single seed, short runs — directional evidence for a thread, not a paper.